In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Business Problem Statement

The objective of this project is to predict residential house sale prices using property characteristics, construction details, and location-related information.

This project applies supervised machine learning regression techniques to analyze housing data and build predictive models capable of estimating house sale prices for unseen properties.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
train_df = pd.read_csv(r'/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
train_df.head()

In [ ]:
print(train_df.info())

In [ ]:
print(train_df.isnull().sum())

In [ ]:
# Fill numerical columns with median
for col in train_df.columns:
    if train_df[col].dtype in ['int64', 'float64']:
        train_df[col] = train_df[col].fillna(train_df[col].median())
# Fill categorical columns with mode
for col in train_df.columns:
    if train_df[col].dtype == 'object':
        train_df[col] = train_df[col].fillna(train_df[col].mode()[0])

In [ ]:
le = LabelEncoder()

for col in train_df.columns:
    if train_df[col].dtype == 'object':
        train_df[col] = le.fit_transform(train_df[col])

In [ ]:
X = train_df.drop(['SalePrice', 'Id'], axis=1)
y = train_df['SalePrice']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
model = RandomForestRegressor(random_state=42)

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :",mae)
print("MSE :",mse)
print("RMSE :",rmse)
print("R2 Score :",r2)

In [ ]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
})
importance = importance.sort_values(by='Importance',ascending=False)

print(importance.head())

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(x='Importance',y='Feature',data=importance.head(10))
plt.title('Top 10 Feature Importances')
plt.show()

In [ ]:
test_df = pd.read_csv(r'/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')
test_df.head()

In [ ]:
house_ids = test_df['Id']

In [ ]:
for col in test_df.columns:
    if test_df[col].dtype in ['int64', 'float64']:
        test_df[col] = test_df[col].fillna(test_df[col].median())

for col in test_df.columns:
    if test_df[col].dtype == 'object':
        test_df[col] = test_df[col].fillna(test_df[col].mode()[0])

In [ ]:
for col in test_df.columns:
    if test_df[col].dtype == 'object':
        test_df[col] = le.fit_transform(test_df[col])

In [ ]:
test_df = test_df.drop('Id', axis=1)

In [ ]:
test_pred = model.predict(test_df)

In [ ]:
submission = pd.DataFrame({
    'Id': house_ids,
    'SalePrice': test_pred
})

submission.head()

In [ ]:
submission.to_csv('submission.csv',index=False)

# Business Conclusion

The Random Forest Regression model successfully predicted residential house sale prices using property characteristics, construction details, and location-related features.

Feature importance analysis revealed that factors such as overall quality, living area size, garage capacity, and neighborhood-related attributes significantly influenced house prices.

The project demonstrates the complete supervised machine learning regression workflow including data preprocessing, feature encoding, model training, evaluation, feature importance analysis, and Kaggle competition submission generation.